# GRDP per Capita — Silver
SQL-only notebook. Run all cells top to bottom in one SQL session.

Scope defaults to 2021–2025. Existing IDs are retained; only new keys receive continuing IDs. Whole-peso rounding, column names, price_basis and existing region mapping are preserved.

Use one writer per target (job maximum concurrent runs = 1). Do not refresh Bronze while this notebook runs. Bronze must contain the complete selected-year snapshot. Missing previous scoped keys fail for review; no automatic deletes.

The team's verified shared geography crosswalk and boundary compatibility, especially NIR, remain prerequisites for Gold joins. This notebook has not been run against Databricks.


## 1. Scope and run metadata
Requires Databricks SQL or Runtime 14.3+ for SQL variables and EXECUTE IMMEDIATE.


In [ ]:
%sql
DECLARE OR REPLACE VARIABLE grdp_start_year INT DEFAULT 2021;
DECLARE OR REPLACE VARIABLE grdp_end_year INT DEFAULT 2025;
DECLARE OR REPLACE VARIABLE grdp_run_id STRING;
DECLARE OR REPLACE VARIABLE grdp_run_ts TIMESTAMP;
DECLARE OR REPLACE VARIABLE grdp_last_id BIGINT;
DECLARE OR REPLACE VARIABLE grdp_schema_sql STRING;
SET VAR grdp_run_id = CONCAT('silver_grdp_pc_', REPLACE(UUID(), '-', ''));
SET VAR grdp_run_ts = CURRENT_TIMESTAMP();
SELECT ASSERT_TRUE(grdp_start_year <= grdp_end_year, 'Invalid year scope.');

## 2. Validate source structure and normalize
Invalid years or rescued rows block the load. Valid years outside the selected scope are excluded. Duplicate keys are not silently removed.


In [ ]:
%sql
SELECT ASSERT_TRUE(
 COUNT_IF(TRY_CAST(year AS INT) IS NULL
   OR NULLIF(TRIM(_rescued_data), '') IS NOT NULL) = 0,
 'Invalid source years or rescued records; repair Bronze first.'
)
FROM pondolens.dev_bronze.grdp_per_capita;

CREATE OR REPLACE TEMP VIEW grdp_pc_stage AS
WITH normalized AS (
 SELECT NULLIF(UPPER(TRIM(region_id)), '') AS source_region_code,
        NULLIF(TRIM(REGEXP_REPLACE(region, r'\s+', ' ')), '') AS region_name,
        TRY_CAST(year AS INT) AS year,
        TRY_CAST(ROUND(TRY_CAST(GRDP_per_capita_amount AS DECIMAL(30,15)),0) AS BIGINT) AS grdp_per_capita,
        LOWER(TRIM(price_basis)) AS source_price_basis,
        NULLIF(LOWER(TRIM(geographic_level)), '') AS source_geographic_level,
        _drive_file_id AS source_drive_file_id, _source_file AS source_file,
        _run_id AS source_ingestion_run_id, _content_hash AS source_content_hash
 FROM pondolens.dev_bronze.grdp_per_capita
 WHERE TRY_CAST(year AS INT) BETWEEN grdp_start_year AND grdp_end_year
)
SELECT CASE WHEN UPPER(region_name) = 'PHILIPPINES' THEN 'PHL'
            WHEN UPPER(region_name) = 'MIMAROPA REGION' THEN 'IV-B'
            ELSE source_region_code END AS region_code,
       region_name, year, grdp_per_capita,
       CASE WHEN source_price_basis = 'at current prices' THEN year
            WHEN source_price_basis = 'at constant 2018 prices' THEN 2018 END AS price_basis,
       CASE WHEN UPPER(region_name) = 'PHILIPPINES' THEN 'national'
            ELSE source_geographic_level END AS geographic_level,
       source_drive_file_id, source_file, source_ingestion_run_id, source_content_hash
FROM normalized;

## 3. Fail on invalid source rows or duplicate keys
Inspect the first two outputs if validation fails. Business key: year, region_code, price_basis, geographic_level.


In [ ]:
%sql
SELECT * FROM grdp_pc_stage WHERE region_code IS NULL OR TRIM(region_code) = ''
 OR region_name IS NULL OR TRIM(region_name) = ''
 OR year IS NULL
 OR grdp_per_capita IS NULL OR grdp_per_capita <= 0
 OR price_basis IS NULL OR (price_basis <> 2018 AND price_basis <> year)
 OR geographic_level IS NULL OR geographic_level NOT IN ('region','national')
 OR (geographic_level = 'national' AND (region_code <> 'PHL' OR UPPER(region_name) <> 'PHILIPPINES'))
 OR (geographic_level = 'region' AND (region_code = 'PHL' OR UPPER(region_name) = 'PHILIPPINES'));
SELECT year, region_code, price_basis, geographic_level, COUNT(*) AS copies
FROM grdp_pc_stage GROUP BY year, region_code, price_basis, geographic_level HAVING COUNT(*) > 1;
SELECT
 ASSERT_TRUE(COUNT(*) > 0, 'No data in selected scope.') AS nonempty,
 ASSERT_TRUE(COUNT_IF(region_code IS NULL OR TRIM(region_code) = ''
 OR region_name IS NULL OR TRIM(region_name) = ''
 OR year IS NULL
 OR grdp_per_capita IS NULL OR grdp_per_capita <= 0
 OR price_basis IS NULL OR (price_basis <> 2018 AND price_basis <> year)
 OR geographic_level IS NULL OR geographic_level NOT IN ('region','national')
 OR (geographic_level = 'national' AND (region_code <> 'PHL' OR UPPER(region_name) <> 'PHILIPPINES'))
 OR (geographic_level = 'region' AND (region_code = 'PHL' OR UPPER(region_name) = 'PHILIPPINES'))) = 0, 'Invalid source key, amount, price basis or geography.') AS valid_data,
 ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT STRUCT(year,region_code,price_basis,geographic_level)),
   'Duplicate source business keys; inspect source files.') AS unique_keys,
 ASSERT_TRUE(COUNT_IF(
 NULLIF(TRIM(source_drive_file_id),'') IS NULL OR NULLIF(TRIM(source_file),'') IS NULL
 OR NULLIF(TRIM(source_ingestion_run_id),'') IS NULL OR NULLIF(TRIM(source_content_hash),'') IS NULL
 ) = 0, 'Missing source lineage.') AS valid_lineage
FROM grdp_pc_stage;

## 4. Prepare target and backfill schema
Existing data and IDs remain. Add only missing lineage columns using information_schema; reruns need no manual ALTER steps.


In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS pondolens.dev_silver;
CREATE TABLE IF NOT EXISTS pondolens.dev_silver.grdp_per_capita (
 grdp_per_capita_id INT, region_code STRING, region_name STRING, year INT,
 grdp_per_capita BIGINT, price_basis INT, geographic_level STRING,
 processing_run_id STRING, processed_at TIMESTAMP,
 source_drive_file_id STRING, source_file STRING,
 source_ingestion_run_id STRING, source_content_hash STRING
) USING DELTA;

SET VAR grdp_schema_sql = (
 WITH required AS (
  SELECT column_name FROM VALUES
   ('source_drive_file_id'), ('source_file'),
   ('source_ingestion_run_id'), ('source_content_hash') AS r(column_name)
 ), missing AS (
  SELECT r.column_name FROM required r
  LEFT ANTI JOIN pondolens.information_schema.columns c
   ON c.table_schema = 'dev_silver' AND c.table_name = 'grdp_per_capita'
   AND c.column_name = r.column_name
 )
 SELECT CASE WHEN COUNT(*) = 0 THEN 'SELECT 1 AS schema_ready'
 ELSE CONCAT('ALTER TABLE pondolens.dev_silver.grdp_per_capita ADD COLUMNS (',
   CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(CONCAT(column_name, ' STRING')))), ')') END
 FROM missing
);
EXECUTE IMMEDIATE grdp_schema_sql;

## 5. Validate existing Silver and missing source keys
Stops before MERGE if target IDs or keys are invalid, or prior keys disappear within the selected scope. Investigate removals rather than keeping stale records silently.


In [ ]:
%sql
SELECT
 ASSERT_TRUE(COUNT_IF(region_code IS NULL OR TRIM(region_code) = ''
 OR region_name IS NULL OR TRIM(region_name) = ''
 OR year IS NULL
 OR grdp_per_capita IS NULL OR grdp_per_capita <= 0
 OR price_basis IS NULL OR (price_basis <> 2018 AND price_basis <> year)
 OR geographic_level IS NULL OR geographic_level NOT IN ('region','national')
 OR (geographic_level = 'national' AND (region_code <> 'PHL' OR UPPER(region_name) <> 'PHILIPPINES'))
 OR (geographic_level = 'region' AND (region_code = 'PHL' OR UPPER(region_name) = 'PHILIPPINES'))) = 0, 'Invalid existing Silver data.') AS valid_target,
 ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT grdp_per_capita_id)
   AND COUNT_IF(grdp_per_capita_id <= 100000000 OR grdp_per_capita_id > 2147483647) = 0,
   'Invalid or duplicate existing IDs.') AS valid_ids,
 ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT STRUCT(year,region_code,price_basis,geographic_level)),
   'Duplicate existing business keys.') AS unique_keys
FROM pondolens.dev_silver.grdp_per_capita;

CREATE OR REPLACE TEMP VIEW grdp_pc_missing_source AS
SELECT t.* FROM pondolens.dev_silver.grdp_per_capita t
LEFT ANTI JOIN grdp_pc_stage s ON t.year = s.year AND t.region_code = s.region_code AND t.price_basis = s.price_basis AND t.geographic_level = s.geographic_level
WHERE t.year BETWEEN grdp_start_year AND grdp_end_year;
SELECT * FROM grdp_pc_missing_source;
SELECT ASSERT_TRUE(COUNT(*) = 0,
 'Existing scoped keys absent from Bronze; review source removal or key correction.')
FROM grdp_pc_missing_source;

## 6. Allocate IDs only to new keys
Capture the current maximum once. Existing IDs never change. Single writer required.


In [ ]:
%sql
SET VAR grdp_last_id = (
 SELECT COALESCE(MAX(CAST(grdp_per_capita_id AS BIGINT)),100000000)
 FROM pondolens.dev_silver.grdp_per_capita
);
CREATE OR REPLACE TEMP VIEW grdp_pc_new_ids AS
SELECT s.year, s.region_code, s.price_basis, s.geographic_level,
 grdp_last_id + ROW_NUMBER() OVER (
  ORDER BY s.year, s.region_code, s.price_basis, s.geographic_level
 ) AS new_id
FROM grdp_pc_stage s
LEFT ANTI JOIN pondolens.dev_silver.grdp_per_capita t ON t.year = s.year AND t.region_code = s.region_code AND t.price_basis = s.price_basis AND t.geographic_level = s.geographic_level;
SELECT ASSERT_TRUE(COUNT_IF(new_id NOT BETWEEN 100000001 AND 2147483647) = 0,
 'INT ID capacity exceeded; migrate to BIGINT before loading.')
FROM grdp_pc_new_ids;

CREATE OR REPLACE TEMP VIEW grdp_pc_merge_source AS
SELECT s.*, t.grdp_per_capita_id AS existing_id, CAST(n.new_id AS INT) AS new_id
FROM grdp_pc_stage s
LEFT JOIN pondolens.dev_silver.grdp_per_capita t ON t.year = s.year AND t.region_code = s.region_code AND t.price_basis = s.price_basis AND t.geographic_level = s.geographic_level
LEFT JOIN grdp_pc_new_ids n
 ON s.year = n.year AND s.region_code = n.region_code
 AND s.price_basis = n.price_basis AND s.geographic_level = n.geographic_level;
SELECT ASSERT_TRUE(COUNT_IF(existing_id IS NULL AND new_id IS NULL) = 0,
 'New record has no allocated ID.') FROM grdp_pc_merge_source;

## 7. Merge
Changed values or lineage update processing metadata. Unchanged records retain metadata. First run backfills lineage; unchanged Bronze reruns should write zero rows.


In [ ]:
%sql
MERGE INTO pondolens.dev_silver.grdp_per_capita t
USING grdp_pc_merge_source s ON t.year = s.year AND t.region_code = s.region_code AND t.price_basis = s.price_basis AND t.geographic_level = s.geographic_level
WHEN MATCHED AND (
 NOT (t.region_name <=> s.region_name)
 OR NOT (t.grdp_per_capita <=> s.grdp_per_capita)
 OR NOT (t.source_drive_file_id <=> s.source_drive_file_id)
 OR NOT (t.source_file <=> s.source_file)
 OR NOT (t.source_ingestion_run_id <=> s.source_ingestion_run_id)
 OR NOT (t.source_content_hash <=> s.source_content_hash)
) THEN UPDATE SET
 t.region_name = s.region_name,
 t.grdp_per_capita = s.grdp_per_capita,
 t.source_drive_file_id = s.source_drive_file_id,
 t.source_file = s.source_file,
 t.source_ingestion_run_id = s.source_ingestion_run_id,
 t.source_content_hash = s.source_content_hash,
 t.processing_run_id = grdp_run_id, t.processed_at = grdp_run_ts
WHEN NOT MATCHED THEN INSERT (
 grdp_per_capita_id, region_code, region_name, year, grdp_per_capita, price_basis, geographic_level, source_drive_file_id, source_file, source_ingestion_run_id, source_content_hash, processing_run_id, processed_at
) VALUES (
 s.new_id, s.region_code, s.region_name, s.year, s.grdp_per_capita, s.price_basis, s.geographic_level, s.source_drive_file_id, s.source_file, s.source_ingestion_run_id, s.source_content_hash, grdp_run_id, grdp_run_ts
);

## 8. Verify scoped preservation and summarize
EXCEPT ALL compares business fields and lineage in both directions. No automatic deletes; older years outside the configured scope remain in Silver.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW grdp_pc_differences AS
WITH actual AS (
 SELECT region_code, region_name, year, grdp_per_capita, price_basis, geographic_level, source_drive_file_id, source_file, source_ingestion_run_id, source_content_hash FROM pondolens.dev_silver.grdp_per_capita
 WHERE year BETWEEN grdp_start_year AND grdp_end_year
), expected AS (SELECT region_code, region_name, year, grdp_per_capita, price_basis, geographic_level, source_drive_file_id, source_file, source_ingestion_run_id, source_content_hash FROM grdp_pc_stage),
missing AS (SELECT * FROM expected EXCEPT ALL SELECT * FROM actual),
extra AS (SELECT * FROM actual EXCEPT ALL SELECT * FROM expected)
SELECT 'MISSING_OR_CHANGED' AS issue, * FROM missing
UNION ALL SELECT 'EXTRA_OR_CHANGED' AS issue, * FROM extra;
SELECT * FROM grdp_pc_differences;
SELECT ASSERT_TRUE(COUNT(*) = 0, 'Post-load source/Silver reconciliation failed.')
FROM grdp_pc_differences;
SELECT ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT grdp_per_capita_id),
 'Post-load duplicate IDs.') FROM pondolens.dev_silver.grdp_per_capita;
SELECT year, COUNT(*) AS records,
 COUNT_IF(processing_run_id = grdp_run_id) AS written_this_run,
 MIN(grdp_per_capita_id) AS min_id, MAX(grdp_per_capita_id) AS max_id
FROM pondolens.dev_silver.grdp_per_capita GROUP BY year ORDER BY year;